# Machine Profiling Notebook

**Goal of this notebook: For you to see how long 4 specified workloads take to run on the machine / development-environment that you are using.**

You do **NOT** need to edit anything in this notebook.

Simply:
1. Click **Run All**; then
2. Wait for the profiling to complete; then
3. Record the **four timing** values display at the end of the notebook, which are: MPC short horizon, MPC long horizon, PPO single environment, and PPO vector environments.

## Colab setup

* When running on **Colab**, this cell installs the pinned course package.
* When running **locally** on your computer, this cell does nothing.

**Note:** installation on Colab can take upto a minute because this notebook needs to install the RL dependencies.

In [1]:
import importlib
import os
from pathlib import Path
import site
import subprocess
import sys
import sysconfig

COURSE_SOURCE_REF = "v0.2.2"
IS_COLAB = "google.colab" in sys.modules
COLAB_CHECKOUT = Path("/content/dream-gym-current")
COURSE_REPOSITORY = "https://gitlab.unimelb.edu.au/dream/dream-gym.git"
package_was_loaded = any(
    name == "dreamgym" or name.startswith("dreamgym.")
    for name in sys.modules
)
setup_diagnostic = None

if IS_COLAB:
    try:
        if COLAB_CHECKOUT.exists():
            if not (COLAB_CHECKOUT / ".git").is_dir():
                raise RuntimeError("unverifiable checkout")
            origin = subprocess.run(
                ("git", "-C", str(COLAB_CHECKOUT), "remote", "get-url", "origin"),
                capture_output=True, text=True, check=True,
            ).stdout.strip()
            if origin != COURSE_REPOSITORY:
                raise RuntimeError("unexpected checkout origin")
            status = subprocess.run(
                ("git", "-C", str(COLAB_CHECKOUT), "status", "--porcelain"),
                capture_output=True, text=True, check=True,
            ).stdout.strip()
            if status:
                raise RuntimeError("checkout has local changes")
            subprocess.run(
                ("git", "-C", str(COLAB_CHECKOUT), "fetch", "--depth", "1", "origin", COURSE_SOURCE_REF),
                capture_output=True, text=True, check=True,
            )
            subprocess.run(
                ("git", "-C", str(COLAB_CHECKOUT), "checkout", "--detach", "FETCH_HEAD"),
                capture_output=True, text=True, check=True,
            )
        else:
            subprocess.run(
                ("git", "clone", "--branch", COURSE_SOURCE_REF,
                 "--single-branch", COURSE_REPOSITORY, str(COLAB_CHECKOUT)),
                capture_output=True, text=True, check=True,
            )
        subprocess.run(
            (sys.executable, "-m", "pip", "install", "-e", ".[rl,mpc]"),
            cwd=COLAB_CHECKOUT, capture_output=True, text=True, check=True,
        )
        site.addsitedir(sysconfig.get_paths()["purelib"])
        os.chdir(COLAB_CHECKOUT)
        importlib.invalidate_caches()
    except (OSError, RuntimeError, subprocess.CalledProcessError):
        setup_diagnostic = "notebook setup failed"
    else:
        setup_diagnostic = (
            "restart runtime before profiling"
            if package_was_loaded else None
        )
else:
    setup_diagnostic = None


In [2]:
if setup_diagnostic is not None:
    raise RuntimeError(
        f"Notebook setup did not complete: {setup_diagnostic}"
    )


## Imports

In [3]:
import dreamgym
dreamgym_source = Path(dreamgym.__file__).resolve()
if IS_COLAB and not dreamgym_source.is_relative_to(COLAB_CHECKOUT.resolve()):
    raise RuntimeError("Dream Gym resolved outside the selected Colab checkout")

import dataclasses

from IPython.display import Markdown, clear_output, display
from dreamgym.profiling.autonomous_driving import (
    ProfileRequest,
    resolve_profile_request,
    run_profile,
)
from dreamgym.profiling.autonomous_driving.notebook import (
    default_notebook_output,
    run_configurable_profile,
)
from dreamgym.profiling.autonomous_driving.progress import (
    TransientNotebookProgress,
)
from dreamgym.profiling.autonomous_driving.records import ResolvedWorkload


## Specify the profiling workloads

In [4]:
def build_student_diagnostic_workload():
    resolved = resolve_profile_request(ProfileRequest(
        preset="quick",
        mpc_prediction_horizon_steps=(20, 80),
        mpc_soft_constraint_modes=("on",),
        rl_algorithms=("PPO",),
        rl_training_steps=(100_000,),
        rl_n_envs=(1, 4),
    ))
    mpc_pairs = tuple(
        (case, execution)
        for case, execution in resolved.workload.cases
        if resolved.suite_for(case, execution) == "mpc"
        and case.semantic_document["mpc"].get("policy") == "steering_rate"
        and case.semantic_document["mpc"].get("road_aware") is False
        and case.semantic_document["mpc"].get("soft_constraints") == "on"
        and case.semantic_document["mpc"].get("prediction_horizon_steps") in (20, 80)
    )
    rl_bindings = tuple(
        binding
        for binding in resolved.rl_cases("ppo_vector_scaling")
        if binding.case_spec.semantic_document["algorithm"].get("name") == "PPO"
        and binding.case_spec.semantic_document["vectorization"].get("kind") == "dummy"
        and binding.case_spec.semantic_document["vectorization"].get("effective_n_envs") in (1, 4)
        and binding.execution_spec.execution_document.get(
            "requested_total_environment_transitions"
        ) == 100_000
    )
    vector_views = tuple(
        view
        for view in resolved.rl_destination_views()
        if view.family == "ppo_vector_scaling"
        and view.requested_transition_target == 100_000
    )
    view_by_identity = {
        view.readable_identity: view
        for view in vector_views
    }
    if (
        len(vector_views) != 2
        or len(view_by_identity) != 2
        or set(view_by_identity) != {"rl-ppo-vector-1", "rl-ppo-vector-4"}
        or tuple(view.readable_identity for view in vector_views)
        != ("rl-ppo-vector-1", "rl-ppo-vector-4")
    ):
        raise ValueError("student diagnostic workload is invalid")
    try:
        selected_views = (
            view_by_identity["rl-ppo-vector-1"],
            view_by_identity["rl-ppo-vector-4"],
        )
    except KeyError as error:
        raise ValueError("student diagnostic workload is invalid") from error
    selected_pairs = mpc_pairs + tuple(
        (binding.case_spec, binding.execution_spec) for binding in rl_bindings
    )
    diagnostic = dataclasses.replace(
        resolved,
        workload=ResolvedWorkload(
            resolved.workload.workload_version,
            selected_pairs,
            schema_version=resolved.workload.schema_version,
        ),
        _core_cases=(),
        _mpc_cases=tuple(case for case, _execution in mpc_pairs),
        _rl_cases=rl_bindings,
        _rl_destination_views=selected_views,
    )
    summary = tuple(
        (
            diagnostic.suite_for(case, execution),
            case.semantic_document["mpc"]["prediction_horizon_steps"]
            if diagnostic.suite_for(case, execution) == "mpc"
            else case.semantic_document["vectorization"]["effective_n_envs"],
        )
        for case, execution in diagnostic.workload.cases
    )
    if summary != (("mpc", 20), ("mpc", 80), ("rl", 1), ("rl", 4)):
        raise ValueError("student diagnostic workload is invalid")
    return diagnostic


def build_student_diagnostic_report_rows(resolved, case_results):
    if len(resolved.workload.cases) != 4 or len(case_results) != 4:
        raise ValueError("student diagnostic report requires four cases")
    expected_identities = tuple(
        (case.semantic_case_id, execution.execution_spec_digest)
        for case, execution in resolved.workload.cases
    )
    actual_identities = tuple(
        (result.case_id, result.execution_spec_digest)
        for result in case_results
    )
    if (
        len(set(actual_identities)) != 4
        or actual_identities != expected_identities
    ):
        raise ValueError(
            "student diagnostic results must match the resolved workload exactly"
        )
    if any(result.status != "completed" for result in case_results):
        raise ValueError("student diagnostic cases must be completed")
    labels_and_units = (
        ("MPC short horizon", "ms per solve"),
        ("MPC long horizon", "ms per solve"),
        ("PPO single environment", "steps per second"),
        ("PPO vector environments", "steps per second"),
    )
    values = (
        case_results[0].metrics["headline"]["median_complete_solver_time_ns"] / 1_000_000,
        case_results[1].metrics["headline"]["median_complete_solver_time_ns"] / 1_000_000,
        case_results[2].metrics["training"]["median_transitions_per_second"],
        case_results[3].metrics["training"]["median_transitions_per_second"],
    )
    return tuple(
        {"label": label, "value": float(value), "unit": unit}
        for (label, unit), value in zip(labels_and_units, values, strict=True)
    )


## Run the profiling

In [5]:
diagnostic_workload = build_student_diagnostic_workload()
diagnostic_progress = TransientNotebookProgress(
    diagnostic_workload,
    display=display,
    clear=lambda: clear_output(wait=True),
)
prior_diagnostic_run = globals().get("diagnostic_run")
diagnostic_run = run_configurable_profile(
    diagnostic_workload,
    output=default_notebook_output(
        "colab" if IS_COLAB else "local",
        "dreamgym-student-diagnostics",
    ),
    runner=run_profile,
    previous=(
        prior_diagnostic_run
        if hasattr(prior_diagnostic_run, "resolved")
        else None
    ),
    fresh=False,
    progress_sink=diagnostic_progress,
    clear_progress=diagnostic_progress.clear,
)
diagnostic_profile = diagnostic_run.result

'Progress 4/4 | elapsed 1:05 | approximate ETA 0:00'

## Display the profiling results

In [6]:
diagnostic_report_rows = build_student_diagnostic_report_rows(
    diagnostic_workload,
    diagnostic_profile.case_results,
)
diagnostic_report_markdown = "\n".join((
    "| Workload | Value | Unit |",
    "|---|---:|---|",
    *(
        f"| {row['label']} | {row['value']:.3g} | {row['unit']} |"
        for row in diagnostic_report_rows
    ),
))
Markdown(diagnostic_report_markdown)

| Workload | Value | Unit |
|---|---:|---|
| MPC short horizon | 3.49 | ms per solve |
| MPC long horizon | 14 | ms per solve |
| PPO single environment | 3.6e+03 | steps per second |
| PPO vector environments | 4.8e+03 | steps per second |

## What you should

The previous cell should display four values:
* MPC short horizon (of length 20 time steps)
* MPC long horizon (of length 80 time steps)
* PPO single environment
* PPO vector environments (with 4 dummy vector environments)

It is expected that:
* MPC with the longer horizon has a longer computation time than with the shorter horizon.
* No clear order between the PPO values. The hope is that the "vector environments" option is faster for some Deep RL algorithms, which it typically is for PPO.